# Comparing Inner Transport Solvers

This tutorial solves the same scattering problem with classic Richardson iteration and PETSc GMRES, then verifies that both methods converge to the same scalar flux.

**Audience:** Users selecting an inner iteration method for a groupset.

**Prerequisites:** Groupsets and a basic fixed-source solve.

## Solve a common reference problem

Classic Richardson exposes source-iteration behavior directly, while GMRES accelerates the linear solve with a Krylov method. The physical model is unchanged between runs. A homogeneous reflecting slab provides an analytic scalar flux of $1/\Sigma_a=5$.

In [ ]:
from mpi4py import MPI
from pyopensn.aquad import GLProductQuadrature1DSlab
from pyopensn.context import Finalize
from pyopensn.mesh import OrthogonalMeshGenerator
from pyopensn.post import VolumePostprocessor
from pyopensn.solver import DiscreteOrdinatesProblem, SteadyStateSourceSolver
from pyopensn.source import VolumetricSource
from pyopensn.xs import MultiGroupXS

rank = MPI.COMM_WORLD.rank

def solve_with(method):
    mesh = OrthogonalMeshGenerator(node_sets=[[i / 20.0 for i in range(21)]]).Execute()
    mesh.SetUniformBlockID(0)
    xs = MultiGroupXS()
    xs.CreateSimpleOneGroup(sigma_t=1.0, c=0.8)
    source = VolumetricSource(block_ids=[0], group_strength=[1.0])
    quadrature = GLProductQuadrature1DSlab(n_polar=8, scattering_order=0)
    problem = DiscreteOrdinatesProblem(
        mesh=mesh,
        num_groups=1,
        groupsets=[
            {
                "groups_from_to": (0, 0),
                "angular_quadrature": quadrature,
                "inner_linear_method": method,
                "l_abs_tol": 1.0e-8,
                "l_max_its": 500,
                "gmres_restart_interval": 30,
            }
        ],
        xs_map=[{"block_ids": [0], "xs": xs}],
        volumetric_sources=[source],
        boundary_conditions=[
            {"name": "zmin", "type": "reflecting"},
            {"name": "zmax", "type": "reflecting"},
        ],
    )
    solver = SteadyStateSourceSolver(problem=problem)
    solver.Initialize()
    solver.Execute()
    average = VolumePostprocessor(problem=problem, value_type="avg")
    average.Execute()
    return float(average.GetValue()[0][0])

richardson_flux = solve_with("classic_richardson")
gmres_flux = solve_with("petsc_gmres")

## Compare the converged solutions

Solver choice should affect the convergence path, not the converged physical solution. Compare the iteration histories printed above as well as the final values below.

In [ ]:
method_difference = abs(richardson_flux - gmres_flux)
analytic_error = max(abs(richardson_flux - 5.0), abs(gmres_flux - 5.0))
if rank == 0:
    print(f"Classic Richardson average flux={richardson_flux:.6e}")
    print(f"PETSc GMRES average flux={gmres_flux:.6e}")
    print(f"Inner-solver flux difference={method_difference:.6e}")
assert method_difference < 1.0e-5
assert analytic_error < 1.0e-5
if "opensn_console" not in globals():
    from IPython import get_ipython
    if get_ipython() is not None:
        Finalize()
        MPI.Finalize()